In [1]:
import pandas as pd
import os
import calculate_worldbank_metrics

In [2]:
SOC_CROSSWALK_FILEPATH = '../01_input_data/labor_codes/2019_to_SOC_Crosswalk.csv'
WORLDBANK_INPUT_URL = '../01_input_data/ai_measurements/worldbank_metrics'
AIOE_FILENAME = 'AIOE_DataAppendix.xlsx'

table_no = 1

# 1. Calculate AI Occupational Exposure (AIOE) per SOC Code

The Artificial Intelligence Occupational Exposure (AIOE) index, introduced by [Felten et al. (2021)](https://sms.onlinelibrary.wiley.com/doi/full/10.1002/smj.3286?__cf_chl_tk=PsCX9B8s7IxRu787dWHte8A8ER.iEVyrfuvdIs9d5e0-1788675243-1.0.1.1-HORPGuasZa2eX7ohUBzY56irDllfBHlxA2oChVTgVHQ) measures the potential exposure of occupations to advances in artificial intelligence (AI) by linking AI capabilities to the abilities required to perform different jobs.

The calculation begins at the ability level, linking ten AI applications to 52 abilities defined by O\*NET. Felten et al. used crowd-sourced ratings collected through Amazon Mechanical Turk (mTurk) to assess the relevance of each AI application to each ability. Summing these ratings produces an AI exposure score, $A_j$, for each ability $j$. Separately, O\*NET provides ratings of the importance and prevalence of each ability within an occupation, denoted by $I_{jk}$ and $L_{jk}$, respectively, for ability $j$ and occupation $k$.

The occupation-level AIOE score is calculated as the weighted average of ability-level AI exposure scores, with the product of each ability's importance and prevalence serving as its weight:

$$
\text{AIOE}_k =
\frac{\sum_{j=1}^{52} A_j L_{jk} I_{jk}}
{\sum_{j=1}^{52} L_{jk} I_{jk}}
$$

Thus, the AIOE reflects how exposed the abilities associated with an occupation are to AI, weighted by their importance and prevalence in that occupation. It measures potential exposure rather than actual AI adoption, automation, or job displacement.

The implementation produces three tables that illustrate the calculation at different stages:

* Table 1 presents the ability-level AI exposure scores, $A_j$. Each row corresponds to one of the 52 O*NET abilities and its associated exposure score, showing how AI exposure is measured before being aggregated to the occupation level.

* Table 2 presents the abilities associated with each occupation, including their importance and prevalence ratings from O*NET. These ratings determine the relative contribution of each ability to an occupation's overall AIOE score.

* Table 3 presents the resulting occupation-level AIOE scores, calculated by combining the ability-level exposure scores with the corresponding O*NET importance and prevalence ratings. Each row corresponds to an occupation and its calculated AI exposure score.



To account for SOC codes without available data, mean imputation will be performed using neighboring SOC codes, defined as occupations sharing the same first five SOC digits. In total, 67 of the 910 SOC codes (7.4%) are missing data and will therefore be imputed using this approach.

In [3]:
soc_crosswalk = pd.read_csv(SOC_CROSSWALK_FILEPATH)
code_title_map = dict(zip(
    soc_crosswalk['O*NET-SOC 2019 Code'], 
    soc_crosswalk['O*NET-SOC 2019 Title']
))

In [4]:
# Show what the ability-level AI exposure dataframe looks like
ability_exposure_map = calculate_worldbank_metrics.get_ability_exposure(
    WORLDBANK_INPUT_URL,
    AIOE_FILENAME,
)

display(
    pd.Series(
        ability_exposure_map,
        name='Ability Level AI Exposure (A_j)',
    )
    .rename_axis('Ability')
    .reset_index()
    .head()
)
print(f'Table {table_no}. Ability-Level AI Exposure Sample')
table_no += 1

,Ability,Ability Level AI Exposure (A_j)
0,Arm-Hand Steadiness,3.46
1,Auditory Attention,6.59
2,Category Flexibility,7.67
3,Control Precision,5.50
4,Deductive Reasoning,7.27


Table 1. Ability-Level AI Exposure Sample


In [5]:
# Show what the occupation-level abilities dataframe looks like
complete_abilities_df = calculate_worldbank_metrics.get_occupation_abilities(
    WORLDBANK_INPUT_URL,
    code_title_map,
)

display(complete_abilities_df.head())
print(f'Table {table_no}. Occupation-Level Abilities Sample')
table_no += 1

,O*NET-SOC Code,Title,Element ID,Element Name,Importance,Level,SOC Prefix
0,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,4.62,4.88,1110
1,11-1011.00,Chief Executives,1.A.1.a.2,Written Comprehension,4.25,4.88,1110
2,11-1011.00,Chief Executives,1.A.1.a.3,Oral Expression,4.50,4.88,1110
3,11-1011.00,Chief Executives,1.A.1.a.4,Written Expression,4.12,4.75,1110
4,11-1011.00,Chief Executives,1.A.1.b.1,Fluency of Ideas,3.88,4.62,1110


Table 2. Occupation-Level Abilities Sample


In [6]:
# Show what the occupation-level AIOE dataframe looks like
aioe_df = calculate_worldbank_metrics.calculate_AIOE(
    WORLDBANK_INPUT_URL,
    AIOE_FILENAME,
    code_title_map,
)

display(aioe_df.head())
print(f'Table {table_no}. Occupation-Level AI Exposure Sample')
table_no += 1

,O*NET-SOC Code,Exposure Weighted,Importance-Weighted Level,AIOE,Standardized AIOE
0,11-1011.00,2581.371106,388.081900,6.651614,1.255658
1,11-1011.03,2091.071768,314.737400,6.643862,1.238338
2,11-1021.00,1915.293738,294.618100,6.500937,0.919019
3,11-1031.00,2181.284959,329.793311,6.614097,1.171838
4,11-2011.00,2057.522618,313.315900,6.566927,1.066452


Table 3. Occupation-Level AI Exposure Sample


# 2. Calculate Complementarity per SOC Code

Pizzinelli et al. (2023) extend the AIOE framework by introducing a complementarity index that measures the extent to which AI supports rather than substitutes human labor. This index is higher for occupations where AI is more likely to enhance human abilities, taking into account both the relevance of AI applications and the broader social and technical contexts that may limit automation.

The authors construct the complementarity index using 11 of the 57 O*NET work context variables together with job zones, grouped into six dimensions. For each occupation in the SOC system, the work context variables provide a score from 0 to 100, which reflects how important or frequent a given task or condition is for that job (e.g., the importance of face-to-face discussions, the frequency of decision-making, or the degree of automation). Job Zone, by contrast, is an ordinal variable ranging from 1 to 5, which indicates the preparation time required to gain the skills for an occupation. In the index, Job Zone values are rescaled by multiplying by 20, producing a 0–100 scale consistent with the other variables. The six grouped dimensions with their work contexts are:

1. *Communication* – Face-to-Face Discussions, Public Speaking

2. *Responsibility* – Responsibility for Outcomes, Responsibility for Others’ Health

3. *Physical Conditions* – Exposure to Outdoor Environments, Physical Proximity to Others

4. *Criticality* – Consequence of Errors, Freedom of Decisions, Frequency of Decisions

5. *Routine* – Degree of Automation\*, Structured vs. Unstructured Work

6. *Skills* – proxied by Job Zone 

For each dimension, an occupation’s score is calculated as the average of its component work context variables. For example, the *Communication* dimension will have its score as the average of the score from Face-to-Face Discussions and Public Speaking. Finally, the complementarity index is obtained by taking the average across the six dimensions, providing a single measure of how strongly AI complements a given occupation.

Table 4 presents the occupation-level complementarity scores calculated using this framework. Each row corresponds to an occupation in the SOC system and its resulting complementarity index, derived by averaging the scores across the six dimensions. The table provides a basis for comparing occupations according to the extent to which AI is expected to complement human labor rather than substitute for it.

---
\*As a note, the degree of automation was inverted because it is the only work context in which higher values indicate lower complementarity with AI. To address this, we created a new variable called degree of freedom, which is derived from the degree of automation. In this formulation, higher scores correspond to greater complementarity with AI. Formally, $$\text{Degree of Freedom} = 100 - \text{Degree of Automation}$$.

In [18]:
complementarity_df = calculate_worldbank_metrics.calculate_complementarity(
    worldbank_input_url=WORLDBANK_INPUT_URL,
    code_title_map=code_title_map
)

display(complementarity_df.head())
print(f'Table {table_no}. Occupation-Level Complementarity Sample')
table_no += 1

,O*NET-SOC Code,Title,Consequence of Error,Degree of Freedom,"Determine Tasks, Priorities and Goals",Face-to-Face Discussions with Individuals and Within Teams,Freedom to Make Decisions,Frequency of Decision Making,Health and Safety of Other Workers,"Outdoors, Exposed to All Weather Conditions",Physical Proximity,Public Speaking,Skills,Work Outcomes and Results of Other Workers,Complementarity
0,11-1011.00,Chief Executives,50.50,69.25,95.00,97.50,98.75,89.75,79.00,28.00,41.75,51.75,100.0,92.50,74.479167
1,11-1011.03,Chief Sustainability Officers,28.75,86.00,86.00,91.75,84.25,66.75,40.75,30.75,44.50,48.25,100.0,71.25,64.916667
2,11-1021.00,General and Operations Managers,38.00,67.25,92.50,96.50,95.25,98.00,88.50,48.25,55.00,36.50,80.0,83.25,73.250000
3,11-2011.00,Advertising and Promotions Managers,29.00,73.50,93.50,89.25,90.75,72.50,36.25,6.25,43.00,45.25,80.0,77.50,61.395833
4,11-2021.00,Marketing Managers,60.25,79.00,96.75,84.50,90.00,81.50,34.50,16.25,36.50,42.25,80.0,67.75,64.104167


Table 6. Occupation-Level Complementarity Sample


# 3. Calculate the Complementarity Adjusted AIOE (i.e., C-AIOE)

We define C-AIOE for some job $j$ as:
$$\text{C-AIOE}_j = \text{AIOE}_j \times \big(1 - (\theta_j - \theta_{\text{MIN}}) \big)$$

This adjustment accounts for the extent to which AI can complement human labor in an occupation. While AIOE measures an occupation's exposure to AI based on the abilities required to perform its tasks, it does not account for whether AI can effectively be used to perform or support those tasks. The complementarity adjustment therefore moderates the original AIOE score based on the occupation's complementarity index, $\theta_j$, relative to the minimum observed value, $\theta_j$.

This is useful because high AI exposure does not necessarily translate into high effective AI applicability. An occupation may require abilities that are highly exposed to AI but operate in contexts where AI is less likely to complement human workers effectively. The adjustment suppresses the AIOE score of such occupations, while occupations with greater AI complementarity retain a larger proportion of their original exposure score. Consequently, C-AIOE provides a more nuanced measure of AI exposure by accounting for both the potential for AI to affect an occupation and the extent to which AI can be effectively integrated into its work.

In [20]:
worldbank_df = (
    pd.concat([aioe_df, complementarity_df], axis=1)
    .drop(columns=['O*NET-SOC Code'])
)

comple = worldbank_df['Complementarity'] / 100
min_comple = comple.min()
worldbank_df['C-AIOE'] = worldbank_df['AIOE'] * (1 - (comple - min_comple))

worldbank_df = worldbank_df[['Title', 'AIOE', 'Standardized AIOE', 'Complementarity', 'C-AIOE']]

worldbank_df.to_csv('../03_output_data/SOC_AIOE_Complementarity.csv', index=False)